# BioPAP Tutorial 2: Create a Pipeline

This tutorial shows how to build and run a preprocessing pipeline with BioPAP's `Pipeline` class. A pipeline applies processing steps in sequence, passing each step's output to the next.

Use this tutorial to:

- Create a pipeline for your data and add processing steps in the order you want them to run.
- Overwrite module configurations from the configuration file.
- Run the complete pipeline and inspect its steps.
- Save the pipeline configuration for reuse, sharing, or batch processing
- Re-create a pipeline from configuration file

### Configuration File

If you don't have a configuration file. Please look at the example configuration file: `./data/pipeline.yaml.example`. You can find example settings of the modules in the module's documentation ([API documentation](../api/)).

If you don't know the correct configurations for the modules, please have a look at [tutorial 1](./1_explore_modules.ipynb).

In [16]:
# Install latest version from local files (only necessary after pulling updated version from Git).
# Restart Jupyter kernel when install has finished.
%pip install -e ..

Obtaining file:///home/matty/ResearchEngineers/bioPAP/BioPAP
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for BioPAP (pyproject.toml) ... done
  Created wheel for BioPAP: filename=biopap-0.0.1-0.editable-py3-none-any.whl size=3011 sha256=085693e36a3d00b9545eed438c44c329290f916add650ad7dedb05a16b19d84e
  Stored in directory: /tmp/pip-ephem-wheel-cache-t1ydx2dq/wheels/21/32/6c/678aeaa2d385c0149fd3ec9bed6c8291c93503a9ed021a1f3d
Successfully built BioPAP
  Attempting uninstall: BioPAP
    Found existing installation: BioPAP 0.0.1
    Uninstalling BioPAP-0.0.1:
      Successfully uninstalled BioPAP-0.0.1
Note: you may need to restart the kernel to use updated packages.


In [17]:
# Some Windows installations require explicitly installing mne (required only once; restart Jupyter kernel after installing).
%pip install mne mne_bids

Note: you may need to restart the kernel to use updated packages.


In [18]:
from biopap.pipeline import PipelineStep, Pipeline

from biopap.modules.load_data import LoadData
from biopap.modules.anonymize import Anonymize
from biopap.modules.rereference import Rereference
from biopap.modules.set_channel_type_and_montage import SetChannelTypeAndMontage
from biopap.modules.eog_channels import EOGChannels
from biopap.modules.blink_detection import BlinkDetection
from biopap.modules.iir_filter import IIRFilter
from biopap.modules.downsample import Downsample
from biopap.modules.segmentation import Segmentation
from biopap.modules.artifact_reject import ArtifactReject
from biopap.modules.baseline_correction import BaselineCorrection
from biopap.modules.save_output import SaveOutput
from biopap.modules.bids import BIDSConversion
from biopap.modules.plot import Plot

In [19]:
import mne
from pathlib import Path
from pprint import pprint

In [20]:
# To avoid issues with local settings and GitHub updates, create a small notebook
# called 'my_config.ipynb' to hold variables that are likely to change locally. 
# In the same folder as this file, there is an example notebook called
# 'example.my_config.ipynb' that can serve as a template.

config_notebook = './data/my_config.ipynb'

if Path(config_notebook).exists():
    %run {config_notebook}
else:
    data_file = './data/example.bdf'
    config_file = './data/pipeline.yaml'

if not Path(config_file).exists():
    raise FileNotFoundError(f"Config file {config_file} does not exist.")

if not Path(data_file).exists():
    raise FileNotFoundError(f"Data file {data_file} does not exist.")

### Creating a Pipeline Instance

Create a new pipeline by instantiating the `Pipeline` class with your data and configuration file:


In [21]:
# Read the data file and load it into memory.
load = LoadData(file_path=data_file)

In [22]:
# Create a new pipeline, loading the raw data and the configuration file
pipe = Pipeline(data=load.raw, config=config_file, verbose='ERROR')

In [23]:
# Inspect the loaded configuration
print("Pipeline Configuration:")
pprint(pipe.config)

Pipeline Configuration:
{'Anonymize': {'_biopap_module': "<class 'biopap.modules.anonymize.Anonymize'>",
               'daysback': 100,
               'experimenter': 'J. Jansen',
               'participant_id': 'AB00123'},
 'BaselineCorrection': {'_biopap_module': '<class '
                                          "'biopap.modules.baseline_correction.BaselineCorrection'>",
                        'baseline': [-0.2, 0]},
 'Downsample': {'_biopap_module': '<class '
                                  "'biopap.modules.downsample.Downsample'>",
                'sfreq': 500},
 'EOGChannels': {'_biopap_module': '<class '
                                   "'biopap.modules.eog_channels.EOGChannels'>",
                 'anode_channels': ['EXG3', 'EXG5'],
                 'cathode_channels': ['EXG4', 'EXG6'],
                 'ref_channels': ['hEOG', 'vEOG']},
 'IIRFilter': {'_biopap_module': '<class '
                                 "'biopap.modules.iir_filter.IIRFilter'>",
               '

### Adding Steps to the Pipeline

A pipeline is made up of processing steps executed in order. Each step's output becomes the next step's input.

**Key points:**
- Do NOT specify input data when adding steps—the pipeline automatically chains outputs
- Settings are read from the configuration file by default
- Steps are not executed until you call `pipe.run()`
- You can reset the pipeline and rebuild it if needed

#### Example 1: Add basic steps with default configuration

In [24]:
# Add processing steps in order
pipe.add_step(Anonymize)
pipe.add_step(Downsample)
pipe.add_step(IIRFilter)
pipe.add_step(Plot)

# Print pipeline overview (steps and their configuration)
print("Pipeline Overview:")
print(*pipe.steps, sep='\n')

2026-10-08 14:17:14,399:INFO:pipeline:Added Anonymize as step #0 with config {'daysback': 100, 'experimenter': 'J. Jansen', 'participant_id': 'AB00123'}
2026-10-08 14:17:14,400:INFO:pipeline:Added Downsample as step #1 with config {'sfreq': 500}
2026-10-08 14:17:14,400:INFO:pipeline:Added IIRFilter as step #2 with config {'h_freq': 35, 'l_freq': 0.1}
2026-10-08 14:17:14,401:INFO:pipeline:Added Plot as step #3 with config {}


Pipeline Overview:
0: Anonymize ← {'daysback': 100, 'experimenter': 'J. Jansen', 'participant_id': 'AB00123'}
1: Downsample ← {'sfreq': 500}
2: IIRFilter ← {'h_freq': 35, 'l_freq': 0.1}
3: Plot ← {}


#### Example 2: Using configuration labels for same module with different settings

When you need to run the same module multiple times with different configurations, use a `cfg_label` to distinguish them in the config file:


In [25]:
# Example: Use Downsample twice with different config labels
pipe.add_step(Downsample, cfg_label='Downsample_500Hz')
pipe.add_step(Downsample, cfg_label='Downsample_1000Hz')  # Different settings in config file

print("Pipeline with multiple Downsample steps:")
print(*pipe.steps, sep='\n')

2026-10-08 14:17:14,407:INFO:pipeline:Added Downsample as step #4 with config {}
2026-10-08 14:17:14,409:INFO:pipeline:Added Downsample as step #5 with config {}


Pipeline with multiple Downsample steps:
0: Anonymize ← {'daysback': 100, 'experimenter': 'J. Jansen', 'participant_id': 'AB00123'}
1: Downsample ← {'sfreq': 500}
2: IIRFilter ← {'h_freq': 35, 'l_freq': 0.1}
3: Plot ← {}
4: Downsample ← {}
5: Downsample ← {}


#### Example 3: Override configuration parameters on-the-fly

You can override configuration file settings by passing a `cfg` dictionary:


In [26]:
# Example: Override configuration with custom parameters
pipe.add_step(IIRFilter, cfg={'h_freq': 50})  # Use 50 Hz instead of config file value

print("IIRFilter with overridden frequency:")
print(*pipe.steps, sep='\n')

2026-10-08 14:17:14,416:INFO:pipeline:Added IIRFilter as step #6 with config {'h_freq': 50, 'l_freq': 0.1}


IIRFilter with overridden frequency:
0: Anonymize ← {'daysback': 100, 'experimenter': 'J. Jansen', 'participant_id': 'AB00123'}
1: Downsample ← {'sfreq': 500}
2: IIRFilter ← {'h_freq': 35, 'l_freq': 0.1}
3: Plot ← {}
4: Downsample ← {}
5: Downsample ← {}
6: IIRFilter ← {'h_freq': 50, 'l_freq': 0.1}


#### Resetting the Pipeline

To clear the pipeline and start over with a different configuration, use `reset()`:


In [27]:
# Clear the pipeline to start fresh
pipe.reset()
print("Pipeline has been reset")

Pipeline has been reset


### Complete Pipeline Example

Build a complete preprocessing pipeline with the recommended processing order:


In [28]:
# Add all processing steps in the recommended order
pipe.add_step(Anonymize)
pipe.add_step(Rereference)
pipe.add_step(SetChannelTypeAndMontage)
pipe.add_step(EOGChannels)
pipe.add_step(IIRFilter)
pipe.add_step(Downsample)
pipe.add_step(Segmentation)
pipe.add_step(BaselineCorrection)
pipe.add_step(SaveOutput)

2026-10-08 14:17:14,434:INFO:pipeline:Added Anonymize as step #0 with config {'daysback': 100, 'experimenter': 'J. Jansen', 'participant_id': 'AB00123'}
2026-10-08 14:17:14,435:INFO:pipeline:Added Rereference as step #1 with config {'ref_channels': ['EXG1', 'EXG2']}
2026-10-08 14:17:14,436:INFO:pipeline:Added SetChannelTypeAndMontage as step #2 with config {'channel_types': {'gsr': ['GSR1', 'GSR2'], 'misc': ['EXG1', 'EXG2', 'EXG3', 'EXG4', 'EXG5', 'EXG6', 'EXG7', 'EXG8', 'Erg1', 'Erg2', 'Plet'], 'resp': ['Resp'], 'temperature': ['Temp']}, 'mne_standard_montage': 'biosemi64'}
2026-10-08 14:17:14,437:INFO:pipeline:Added EOGChannels as step #3 with config {'anode_channels': ['EXG3', 'EXG5'], 'cathode_channels': ['EXG4', 'EXG6'], 'ref_channels': ['hEOG', 'vEOG']}
2026-10-08 14:17:14,437:INFO:pipeline:Added IIRFilter as step #4 with config {'h_freq': 35, 'l_freq': 0.1}
2026-10-08 14:17:14,438:INFO:pipeline:Added Downsample as step #5 with config {'sfreq': 500}
2026-10-08 14:17:14,438:INFO:p

### Inspect the  pipeline

In [29]:
# Display the complete pipeline
pipe.inspect()

00. Anonymize
    daysback: 100
    experimenter: 'J. Jansen'
    participant_id: 'AB00123'

01. Rereference *
    ref_channels: ['EXG1', 'EXG2']

02. SetChannelTypeAndMontage *
    channel_types:
      gsr: ['GSR1', 'GSR2']
      misc: ['EXG1', 'EXG2', 'EXG3', 'EXG4', 'EXG5', 'EXG6', 'EXG7', 'EXG8', 'Erg1', 'Erg2', 'Plet']
      resp: ['Resp']
      temperature: ['Temp']
    mne_standard_montage: 'biosemi64'

03. EOGChannels *
    anode_channels: ['EXG3', 'EXG5']
    cathode_channels: ['EXG4', 'EXG6']
    ref_channels: ['hEOG', 'vEOG']

04. IIRFilter *
    h_freq: 35
    l_freq: 0.1

05. Downsample *
    sfreq: 500

06. Segmentation *
    start: -0.2
    end: 0.8
    reference_markers:
      range: '1-69'
    stim_channel: 'Status'

07. BaselineCorrection *
    baseline: [-0.2, 0]

08. SaveOutput
    export_formats: 'fif'
    output_file: './processed.fif'
    overwrite: True

(modules marked * alter their input data)


### Running the Pipeline

Once you've added all the desired steps, execute the entire pipeline with a single command:


In [30]:
# Run the entire pipeline sequentially
pipe.run()

print("Pipeline execution complete!")

2026-10-08 14:17:14,861:INFO:pipeline:Running (0) Anonymize [CBN1BD071212-Cropped]
2026-10-08 14:17:14,864:INFO:pipeline:Running (1) Rereference < {'ref_channels': ['EXG1', 'EXG2']} [CBN1BD071212-Cropped]
2026-10-08 14:17:15,726:INFO:pipeline:Running (2) SetChannelTypeAndMontage < {'channel_types': {'gsr': ['GSR1', 'GSR2'], 'misc': ['EXG1', 'EXG2', 'EXG3', 'EXG4', 'EXG5', 'EXG6', 'EXG7', 'EXG8', 'Erg1', 'Erg2', 'Plet'], 'resp': ['Resp'], 'temperature': ['Temp']}, 'mne_standard_montage': 'biosemi64'} [CBN1BD071212-Cropped]
2026-10-08 14:17:15,732:INFO:pipeline:Running (3) EOGChannels < {'anode_channels': ['EXG3', 'EXG5'], 'cathode_channels': ['EXG4', 'EXG6'], 'ref_channels': ['hEOG', 'vEOG']} [CBN1BD071212-Cropped]
2026-10-08 14:17:16,621:INFO:pipeline:Running (4) IIRFilter < {'h_freq': 35, 'l_freq': 0.1} [CBN1BD071212-Cropped]
2026-10-08 14:17:21,182:INFO:pipeline:Running (5) Downsample < {'sfreq': 500} [CBN1BD071212-Cropped]
2026-10-08 14:17:33,790:INFO:pipeline:Running (6) Segmentati

Pipeline execution complete!


### Saving the Pipeline

When the pipeline ran without errors, you can save the pipeline with:


In [ ]:
pipe.to_config('./config.yaml')

### Saving changed configuration

Should you add a step that includes ad-hoc configuration to your pipeline, the extra or updated configuration settings are not part of the configuration file you created the pipeline with. To make sure you save the most up to date version of the configuration, call the pipeline's 'to_config()` method.

For instance, in the example configuration, the ArtifactReject-module has a value of 100 for the variable 'interval_ms', but when adding the module to the pipeline, the value could be updated to 200:

In [33]:
pipe.reset()

pipe.add_step(Anonymize)
pipe.add_step(Rereference)
pipe.add_step(SetChannelTypeAndMontage)
pipe.add_step(EOGChannels)
pipe.add_step(IIRFilter)
pipe.add_step(Downsample, cfg={'sfreq': 400})
pipe.add_step(BlinkDetection)
pipe.add_step(Segmentation)
pipe.add_step(ArtifactReject, cfg={'interval_ms':200})
pipe.add_step(BaselineCorrection)
pipe.add_step(SaveOutput)

pipe.to_config('./updated_config.yaml')

2026-10-08 14:24:20,179:INFO:pipeline:Added Anonymize as step #0 with config {'daysback': 100, 'experimenter': 'J. Jansen', 'participant_id': 'AB00123'}
2026-10-08 14:24:20,180:INFO:pipeline:Added Rereference as step #1 with config {'ref_channels': ['EXG1', 'EXG2']}
2026-10-08 14:24:20,181:INFO:pipeline:Added SetChannelTypeAndMontage as step #2 with config {'channel_types': {'gsr': ['GSR1', 'GSR2'], 'misc': ['EXG1', 'EXG2', 'EXG3', 'EXG4', 'EXG5', 'EXG6', 'EXG7', 'EXG8', 'Erg1', 'Erg2', 'Plet'], 'resp': ['Resp'], 'temperature': ['Temp']}, 'mne_standard_montage': 'biosemi64'}
2026-10-08 14:24:20,181:INFO:pipeline:Added EOGChannels as step #3 with config {'anode_channels': ['EXG3', 'EXG5'], 'cathode_channels': ['EXG4', 'EXG6'], 'ref_channels': ['hEOG', 'vEOG']}
2026-10-08 14:24:20,182:INFO:pipeline:Added IIRFilter as step #4 with config {'h_freq': 35, 'l_freq': 0.1}
2026-10-08 14:24:20,183:INFO:pipeline:Added Downsample as step #5 with config {'sfreq': 400}
2026-10-08 14:24:20,183:INFO:p

Please note that in the config file, the steps will likely end up in the file sorted by step name, rather than by step order. This is to be expected, as the Yaml-format does not guarantee the order of its elements. However, when the pipeline reads back the config file, it will sort the steps by step order, recreating the intended order or processing.

---

# Re-creating pipeline from config file

By loading a saved config file into a new `Pipeline`-instance, you automatically create the complete pipeline with all the specified settings.

In [34]:
cfg_file = './updated_config.yaml'

new_pipe = Pipeline.from_config(
    config=cfg_file,
    input_file=data_file,
    subject_id='ABC123'
)

new_pipe.inspect()

2026-10-08 14:25:51,036:INFO:pipeline:Loaded '../../data/CBN1BD071212-Cropped.bdf'
2026-10-08 14:25:51,036:INFO:pipeline:Added Anonymize as step #0 with config {'daysback': 100, 'experimenter': 'J. Jansen', 'participant_id': 'AB00123'}
2026-10-08 14:25:51,037:INFO:pipeline:Added ArtifactReject as step #8 with config {'interval_ms': 200}
2026-10-08 14:25:51,037:INFO:pipeline:Added BaselineCorrection as step #9 with config {'baseline': [-0.2, 0]}
2026-10-08 14:25:51,037:INFO:pipeline:Added BlinkDetection as step #6 with config {}
2026-10-08 14:25:51,037:INFO:pipeline:Added Downsample as step #5 with config {'sfreq': 400}
2026-10-08 14:25:51,038:INFO:pipeline:Added EOGChannels as step #3 with config {'anode_channels': ['EXG3', 'EXG5'], 'cathode_channels': ['EXG4', 'EXG6'], 'ref_channels': ['hEOG', 'vEOG']}
2026-10-08 14:25:51,038:INFO:pipeline:Added IIRFilter as step #4 with config {'h_freq': 35, 'l_freq': 0.1}
2026-10-08 14:25:51,038:INFO:pipeline:Added Rereference as step #1 with config

00. Anonymize
    daysback: 100
    experimenter: 'J. Jansen'
    participant_id: 'AB00123'

01. Rereference *
    ref_channels: ['EXG1', 'EXG2']

02. SetChannelTypeAndMontage *
    channel_types:
      gsr: ['GSR1', 'GSR2']
      misc: ['EXG1', 'EXG2', 'EXG3', 'EXG4', 'EXG5', 'EXG6', 'EXG7', 'EXG8', 'Erg1', 'Erg2', 'Plet']
      resp: ['Resp']
      temperature: ['Temp']
    mne_standard_montage: 'biosemi64'

03. EOGChannels *
    anode_channels: ['EXG3', 'EXG5']
    cathode_channels: ['EXG4', 'EXG6']
    ref_channels: ['hEOG', 'vEOG']

04. IIRFilter *
    h_freq: 35
    l_freq: 0.1

05. Downsample *
    sfreq: 400

06. BlinkDetection *

07. Segmentation *
    end: 0.8
    reference_markers:
      range: '1-69'
    start: -0.2
    stim_channel: 'Status'

08. ArtifactReject *
    interval_ms: 200

09. BaselineCorrection *
    baseline: [-0.2, 0]

10. SaveOutput
    export_formats: 'fif'
    output_file: './processed.fif'
    overwrite: True

(modules marked * alter their input data)


## A word on subject_id

The subject ID can be used to differentiate between outputs for different users when running the pipeline in batches. When you use the string `{subject_id}` in the value of any string-type config variable, it will be automatically replaced by the value of subject_id. So, for instance, by defining a pipeline-step thus:

```python
SaveOutput(
    raw=baseline.processed,
    export_formats='fif',
    output_file='./partially_processed_{subject_id}.fif',
    overwrite=True)
```
running the pipeline with `subject_id="A123"` will result in an outputfile `partially_processed_A123.fif`.

If you do not specify a subject_id when creating the pipeline, the program will use the input file's name without the suffix. It will first try to read this from the 'filenames' attribute from the file's metadata, and if that is empty, use the actual file's path.